In [1]:
print("Hello, World!")

Hello, World!


In [7]:
import os
import re
import json
import time
import random
from pathlib import Path

import requests
from dotenv import load_dotenv
from google import genai
from datasets import load_dataset

load_dotenv()

# ---------- config ----------

MODEL_ID = "gemini-3.6-flash"
TEMPERATURE = 0.0
MAX_OUTPUT_TOKENS = 512
MAX_HOPS = 4
RETRY_BADCALLS = True          # was False — now lets true parse failures get one retry
RETRY_ATTEMPTS = 2
RETRY_BACKOFF_S = 5
PACE_SECONDS = 6.5              # keeps you under 10 RPM with a small buffer

N_QUESTIONS = 200
SEED = 42

OUT_PATH = Path("results_baseline_gemini.jsonl")


# ---------- wikipedia tool ----------

SENTENCE_SPLIT = re.compile(r"(?<=[.!?])\s+")
WIKI_API = "https://en.wikipedia.org/w/api.php"


class WikiEnv:
    def __init__(self, intro_sentences=5):
        self.intro_sentences = intro_sentences
        self._sentences = []
        self._lookup_keyword = None
        self._lookup_pos = 0
        self.headers = {"User-Agent": "ReActLocalAgent/1.0 (personal research project)"}
        self._search_cache = {}

    def search(self, entity):
        if entity in self._search_cache:
            cached = self._search_cache[entity]
            self._sentences = cached["sentences"]
            self._lookup_keyword = None
            self._lookup_pos = 0
            return cached["intro"]

        page = self._fetch_extract(entity)
        if page is None:
            similar = self._fetch_suggestions(entity)
            self._sentences = []
            self._lookup_keyword = None
            result = f"Could not find [{entity}]. Similar: {similar}" if similar else f"Could not find [{entity}]."
            self._search_cache[entity] = {"sentences": [], "intro": result}
            return result

        title, extract = page
        self._sentences = [s for s in SENTENCE_SPLIT.split(extract) if s.strip()]
        self._lookup_keyword = None
        self._lookup_pos = 0
        intro = " ".join(self._sentences[: self.intro_sentences]) or f"[{title}] found but has no readable text."
        self._search_cache[entity] = {"sentences": self._sentences, "intro": intro}
        return intro

    def lookup(self, keyword):
        if not self._sentences:
            return "No page open. Use Search first."
        if keyword != self._lookup_keyword:
            self._lookup_keyword = keyword
            self._lookup_pos = 0
        matches = [i for i, s in enumerate(self._sentences) if keyword.lower() in s.lower()]
        if not matches:
            return f"No mentions of [{keyword}] found."
        remaining = [i for i in matches if i >= self._lookup_pos]
        if not remaining:
            return f"No more results for [{keyword}]."
        idx = remaining[0]
        self._lookup_pos = idx + 1
        result_num = matches.index(idx) + 1
        return f"(Result {result_num}/{len(matches)}) {self._sentences[idx]}"

    def _fetch_extract(self, title):
        params = {"action": "query", "prop": "extracts", "explaintext": 1,
                  "redirects": 1, "titles": title, "format": "json"}
        resp = requests.get(WIKI_API, params=params, headers=self.headers, timeout=15)
        resp.raise_for_status()
        pages = resp.json().get("query", {}).get("pages", {})
        for page_id, page in pages.items():
            if page_id == "-1" or "missing" in page:
                return None
            extract = page.get("extract", "")
            if extract.strip():
                return page.get("title", title), extract
        return None

    def _fetch_suggestions(self, query, limit=5):
        params = {"action": "query", "list": "search", "srsearch": query,
                  "srlimit": limit, "format": "json"}
        resp = requests.get(WIKI_API, params=params, headers=self.headers, timeout=15)
        resp.raise_for_status()
        results = resp.json().get("query", {}).get("search", [])
        return ", ".join(r["title"] for r in results)


# ---------- gemini client with key rotation ----------

class LLMResponse:
    def __init__(self, text, prompt_tokens, completion_tokens):
        self.text = text
        self.prompt_tokens = prompt_tokens
        self.completion_tokens = completion_tokens


class KeysExhaustedError(Exception):
    pass


def load_api_keys():
    raw = os.environ.get("GEMINI_API_KEYS", "")
    keys = [k.strip() for k in raw.split(",") if k.strip()]
    if not keys:
        raise RuntimeError("set GEMINI_API_KEYS in your .env, comma-separated")
    return keys


class GeminiClient:
    def __init__(self, model_id=MODEL_ID, temperature=TEMPERATURE, max_output_tokens=MAX_OUTPUT_TOKENS,
                 retry_attempts=RETRY_ATTEMPTS, retry_backoff_s=RETRY_BACKOFF_S, pace_seconds=PACE_SECONDS):
        self.api_keys = load_api_keys()
        self.key_idx = 0
        self.model_id = model_id
        self.temperature = temperature
        self.max_output_tokens = max_output_tokens
        self.retry_attempts = retry_attempts
        self.retry_backoff_s = retry_backoff_s
        self.pace_seconds = pace_seconds
        self.client = genai.Client(api_key=self.api_keys[self.key_idx])

    def _rotate_key(self):
        self.key_idx += 1
        if self.key_idx >= len(self.api_keys):
            return False
        self.client = genai.Client(api_key=self.api_keys[self.key_idx])
        print(f"key #{self.key_idx} hit quota, switching to key #{self.key_idx + 1}")
        return True

    def _classify_error(self, err):
        msg = str(err).lower()
        if "per minute" in msg or "rpm" in msg:
            return "rpm"
        if "per day" in msg or "rpd" in msg or "resource_exhausted" in msg or "quota" in msg:
            return "rpd"
        return "other"

    def complete(self, prompt, stop=None):
        generation_config = {"temperature": self.temperature, "max_output_tokens": self.max_output_tokens}
        if stop:
            generation_config["stop_sequences"] = stop

        keys_tried = 0
        last_err = None
        while keys_tried <= len(self.api_keys):
            for attempt in range(1, self.retry_attempts + 1):
                try:
                    time.sleep(self.pace_seconds)
                    interaction = self.client.interactions.create(
                        model=self.model_id, input=prompt, generation_config=generation_config,
                    )
                    usage = interaction.usage
                    return LLMResponse(
                        text=interaction.output_text,
                        # FIX: the Interactions API usage object uses prompt_tokens /
                        # completion_tokens, not total_input_tokens / total_output_tokens.
                        # The old attribute names silently fell back to 0 every call.
                        prompt_tokens=getattr(usage, "prompt_tokens", 0),
                        completion_tokens=getattr(usage, "completion_tokens", 0),
                    )
                except Exception as e:
                    last_err = e
                    kind = self._classify_error(e)
                    if kind == "rpm":
                        # FIX: treat an RPM error that survives the wait loop as a quota
                        # issue too, so it falls through to key rotation below instead of
                        # being misreported as "not a quota issue" and killing the run.
                        time.sleep(10)  # wait out the per-minute window, same key
                        if attempt == self.retry_attempts:
                            break  # give up on this key, go rotate
                        continue
                    if kind == "rpd":
                        break  # no point retrying, go rotate
                    if attempt < self.retry_attempts:
                        time.sleep(self.retry_backoff_s * attempt)

            if self._classify_error(last_err) in ("rpd", "rpm"):
                keys_tried += 1
                if not self._rotate_key():
                    raise KeysExhaustedError(str(last_err))
                continue
            raise RuntimeError(f"Gemini call failed, not a quota issue: {last_err}")

        raise KeysExhaustedError(str(last_err))


# ---------- prompt ----------

INSTRUCTIONS = """Answer the question by interleaving numbered Thought, Action, and Observation steps.
Action is one of three types:
(1) Search[entity] - looks up the entity on Wikipedia, returns the opening paragraph if it exists, or similar titles if not.
(2) Lookup[keyword] - returns the next sentence on the current page containing keyword.
(3) Finish[answer] - gives the final answer and ends the task.
Here are some examples.
"""

FEWSHOT_EXAMPLES = """
Question: What is the nationality of the director of the 1994 film that won the Palme d'Or at Cannes?
Thought 1: I need to find which film won the Palme d'Or in 1994, then its director, then his nationality.
Action 1: Search[1994 Cannes Film Festival]
Observation 1: The 1994 Cannes Film Festival awarded the Palme d'Or to Pulp Fiction, directed by Quentin Tarantino.
Thought 2: The director is Quentin Tarantino. Now his nationality.
Action 2: Search[Quentin Tarantino]
Observation 2: Quentin Jerome Tarantino is an American film director, screenwriter, producer, and actor.
Thought 3: He's American.
Action 3: Finish[American]

Question: The university where the inventor of the World Wide Web earned his degree is in which city?
Thought 1: Find who invented the Web, then where he studied.
Action 1: Search[World Wide Web]
Observation 1: The World Wide Web was invented by English computer scientist Tim Berners-Lee.
Thought 2: Now find where he studied.
Action 2: Search[Tim Berners-Lee]
Observation 2: Tim Berners-Lee studied physics at The Queen's College, Oxford, graduating in 1976.
Thought 3: Oxford is both the college and the city.
Action 3: Finish[Oxford]
""".strip()


def build_prompt(question, trajectory):
    return f"{INSTRUCTIONS}\n\n{FEWSHOT_EXAMPLES}\n\nQuestion: {question}\n{trajectory}"


# ---------- react agent ----------

ACTION_RE = re.compile(r"(\w+)\[(.*?)\]", re.DOTALL)


class ReActAgent:
    def __init__(self, llm, max_hops=MAX_HOPS, retry_badcalls=RETRY_BADCALLS):
        self.llm = llm
        self.max_hops = max_hops
        self.retry_badcalls = retry_badcalls

    def _split_thought_and_action(self, generated, hop_idx):
        """
        Returns (thought, action_str, was_badcall).

        FIX: the original code required the literal substring "\nAction N:"
        to be present, which broke whenever the model:
          - opened directly with "Action N: Search[...]" (no leading newline
            since it's the first thing generated), or
          - skipped the Thought/Action labels entirely and chained bare
            Search[...]Search[...] calls together.
        This version searches for the "Action N:" label anywhere in the text,
        and falls back to grabbing the first bracketed action if no label is
        present at all, instead of discarding the whole hop.
        """
        action_label_re = re.compile(rf"Action\s*{hop_idx}\s*:\s*", re.IGNORECASE)

        m = action_label_re.search(generated)
        if m:
            thought = generated[: m.start()].strip()
            action_str = generated[m.end():].strip()
            return thought, action_str, False

        bare_match = ACTION_RE.search(generated)
        if bare_match:
            thought = generated[: bare_match.start()].strip()
            action_str = generated[bare_match.start():].strip()
            return thought, action_str, True  # salvaged, but flagged as non-compliant

        thought = generated.split("\n")[0].strip()
        return thought, "", True  # true parse failure, no action found at all

    def run(self, question_id, question, gold_answer):
        env = WikiEnv()
        trajectory_text = ""
        hops = []
        predicted_answer = None
        stopped_reason = "max_hops"
        n_calls = 0
        n_badcalls = 0
        total_tokens = 0

        for hop_idx in range(1, self.max_hops + 1):
            prompt = build_prompt(question, trajectory_text) + f"Thought {hop_idx}:"
            response = self.llm.complete(prompt, stop=[f"\nObservation {hop_idx}:"])
            n_calls += 1
            total_tokens += response.prompt_tokens + response.completion_tokens

            generated = response.text.strip()
            thought, action_str, was_badcall = self._split_thought_and_action(generated, hop_idx)

            if was_badcall:
                n_badcalls += 1

            if was_badcall and not action_str:
                # true parse failure: no action found at all
                if not self.retry_badcalls:
                    stopped_reason = "parse_error"
                    hops.append({"hop": hop_idx, "thought": thought, "action_type": "PARSE_ERROR",
                                 "action_arg": "", "observation": "badcall, retry disabled", "badcall": True})
                    break
                n_calls += 1
                retry_prompt = build_prompt(question, trajectory_text) + f"Thought {hop_idx}: {thought}\nAction {hop_idx}:"
                retry = self.llm.complete(retry_prompt, stop=["\n"])
                total_tokens += retry.prompt_tokens + retry.completion_tokens
                action_str = retry.text.strip()
                if not action_str:
                    stopped_reason = "parse_error"
                    hops.append({"hop": hop_idx, "thought": thought, "action_type": "PARSE_ERROR",
                                 "action_arg": "", "observation": "badcall, retry exhausted", "badcall": True})
                    break

            action_match = ACTION_RE.search(action_str)
            if not action_match:
                stopped_reason = "parse_error"
                hops.append({"hop": hop_idx, "thought": thought, "action_type": "PARSE_ERROR",
                             "action_arg": "", "observation": f"could not parse: {action_str!r}", "badcall": was_badcall})
                break

            action_type = action_match.group(1).strip().capitalize()
            action_arg = action_match.group(2).strip()

            if action_type == "Finish":
                predicted_answer = action_arg
                stopped_reason = "finished"
                hops.append({"hop": hop_idx, "thought": thought, "action_type": action_type,
                             "action_arg": action_arg, "observation": "", "badcall": was_badcall})
                break

            if action_type == "Search":
                observation = env.search(action_arg)
            elif action_type == "Lookup":
                observation = env.lookup(action_arg)
            else:
                observation = f"unrecognized action type: {action_type}"

            hops.append({"hop": hop_idx, "thought": thought, "action_type": action_type,
                         "action_arg": action_arg, "observation": observation, "badcall": was_badcall})
            trajectory_text += f"Thought {hop_idx}: {thought}\nAction {hop_idx}: {action_type}[{action_arg}]\nObservation {hop_idx}: {observation}\n"

        return {
            "question_id": question_id, "question": question, "gold_answer": gold_answer,
            "predicted_answer": predicted_answer, "hops": hops, "stopped_reason": stopped_reason,
            "n_calls": n_calls, "n_badcalls": n_badcalls, "total_tokens": total_tokens,
        }


# ---------- dataset ----------

def sample_hotpotqa(n=N_QUESTIONS, seed=SEED, hard_only=True):
    ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
    pool = [q for q in ds if q["level"] == "hard"] if hard_only else list(ds)
    if hard_only and len(pool) < n:
        print(f"only {len(pool)} hard questions available, using full pool")
        pool = list(ds)
    rng = random.Random(seed)
    sample = rng.sample(pool, min(n, len(pool)))
    return [{"id": q["id"], "question": q["question"], "answer": q["answer"]} for q in sample]


# ---------- resumable batch runner ----------

def load_done_ids(path):
    if not path.exists():
        return set()
    with open(path) as f:
        return {json.loads(line)["question_id"] for line in f if line.strip()}


def run_batch(agent, questions, out_path):
    done_ids = load_done_ids(out_path)
    remaining = [q for q in questions if q["id"] not in done_ids]
    print(f"{len(done_ids)} already done, {len(remaining)} remaining")

    with open(out_path, "a") as f:
        for i, q in enumerate(remaining):
            try:
                result = agent.run(q["id"], q["question"], q["answer"])
                f.write(json.dumps(result) + "\n")
                f.flush()
                print(f"[{i + 1}/{len(remaining)}] {q['id']} -> {result['predicted_answer']}")
            except KeysExhaustedError:
                print("all API keys exhausted - results saved so far, rerun this cell "
                      "with fresh/reset keys to pick up where this left off")
                break
            except Exception as e:
                print(f"question {q['id']} failed with a non-quota error, skipping: {e}")
                continue

    print(f"done for now. {len(load_done_ids(out_path))}/{len(questions)} total complete in {out_path}")


# ---------- results summary for your write-up ----------

def summarize_results(path=OUT_PATH):
    """
    Prints the numbers you'd actually want to report: exact-match accuracy,
    average hops/tokens/calls per question, and a breakdown of how
    trajectories ended (finished vs. max_hops vs. parse_error).
    Doesn't touch the JSONL — safe to re-run anytime.
    """
    rows = []
    with open(path) as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))

    if not rows:
        print("no results yet")
        return

    n = len(rows)

    def normalize(s):
        return re.sub(r"[^a-z0-9]", "", (s or "").lower())

    correct = sum(
        1 for r in rows
        if r["predicted_answer"] is not None
        and normalize(r["predicted_answer"]) == normalize(r["gold_answer"])
    )

    stopped_counts = {}
    for r in rows:
        stopped_counts[r["stopped_reason"]] = stopped_counts.get(r["stopped_reason"], 0) + 1

    avg_hops = sum(len(r["hops"]) for r in rows) / n
    avg_calls = sum(r["n_calls"] for r in rows) / n
    avg_badcalls = sum(r["n_badcalls"] for r in rows) / n
    avg_tokens = sum(r["total_tokens"] for r in rows) / n
    total_tokens = sum(r["total_tokens"] for r in rows)

    print(f"n questions:        {n}")
    print(f"exact match acc:    {correct}/{n} = {correct / n:.1%}")
    print(f"avg hops/question:  {avg_hops:.2f}")
    print(f"avg calls/question: {avg_calls:.2f}")
    print(f"avg badcalls/q:     {avg_badcalls:.2f}")
    print(f"avg tokens/q:       {avg_tokens:.1f}")
    print(f"total tokens:       {total_tokens}")
    print("stop reasons:")
    for reason, count in sorted(stopped_counts.items(), key=lambda kv: -kv[1]):
        print(f"  {reason:12s} {count:4d}  ({count / n:.1%})")


In [8]:
llm = GeminiClient()
agent = ReActAgent(llm)
questions = sample_hotpotqa()

In [9]:
run_batch(agent, questions, OUT_PATH)


4 already done, 196 remaining
[1/196] 5ab985eb554299131ca42360 -> Greyia
[2/196] 5a8753d95542994846c1cd63 -> Yes
[3/196] 5ab97d0a5542996be202051e -> John André
[4/196] 5adef1b35542993a75d263af -> Salma Hayek
[5/196] 5a73b55855429978a71e9086 -> Gatwick Airport
key #1 hit quota, switching to key #2
[6/196] 5ae80919554299540e5a56f6 -> None
[7/196] 5ae361605542994393b9e69b -> None
[8/196] 5ac3a76e554299741d48a2be -> U2
[9/196] 5a7557d75542992d0ec05f68 -> Laurie Metcalf
[10/196] 5a8d12ca5542994ba4e3dbe2 -> Nairobi
[11/196] 5ab7f6d35542993667794070 -> Cyclic Defrost
key #2 hit quota, switching to key #3
[12/196] 5ac3d9135542995c82c4ac4c -> 629
key #3 hit quota, switching to key #4
question 5ae129115542990adbacf722 failed with a non-quota error, skipping: Gemini call failed, not a quota issue: Error code: 403 - {'error': {'message': 'Your project has been denied access. Please contact support.', 'code': 'permission_denied'}}
question 5a86769c5542994775f60776 failed with a non-quota error, ski

KeyboardInterrupt: 

In [ ]:
summarize_results(OUT_PATH)